## Supabase connect get data

In [59]:
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd
from datetime import datetime, timedelta, date
import pandas as pd
import holidays

import numpy as np


# Load env
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print(SUPABASE_URL)

response = supabase.rpc("delete_outlier_data_by_year", {"tahun_param": 2026}).execute()
print(response.data)

https://extlxiwpcbzqaalpopqn.supabase.co


03/03/2026 01:55:53 PM - HTTP Request: POST https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/rpc/delete_outlier_data_by_year "HTTP/2 204 No Content"


[]


In [60]:
# Fungsi untuk mengambil data dengan paginasi dan filter tahun
def fetch_data_with_pagination_and_year(table_name, year, limit=1000):
    offset = 0
    all_data = []
    
    while True:
        # Ambil data per halaman (limit) dengan filter tahun menggunakan format tanggal
        response = supabase.table(table_name).select("*").filter("tanggal", "gte", f"{year}-01-01").filter("tanggal", "lt", f"{year + 1}-01-01").range(offset, offset + limit - 1).execute()
        
        # Cek apakah ada data yang diterima
        if len(response.data) == 0:
            break
        
        # Gabungkan data yang sudah diambil
        all_data.extend(response.data)
        
        # Update offset untuk mengambil data berikutnya
        offset += limit
    
    return all_data

# Ambil data dari tabel harga_beras untuk tahun 2025
data_harga_beras = fetch_data_with_pagination_and_year("harga_beras", 2026)
data_harga_beras = pd.DataFrame(data_harga_beras)

# Tampilkan jumlah data yang diambil
print(f"Jumlah data untuk tahun 2026 yang diambil: {len(data_harga_beras)}")
data_harga_beras

03/03/2026 01:55:53 PM - HTTP Request: GET https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/harga_beras?select=%2A&tanggal=gte.2026-01-01&tanggal=lt.2027-01-01&offset=0&limit=1000 "HTTP/2 200 OK"
03/03/2026 01:55:53 PM - HTTP Request: GET https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/harga_beras?select=%2A&tanggal=gte.2026-01-01&tanggal=lt.2027-01-01&offset=1000&limit=1000 "HTTP/2 200 OK"
03/03/2026 01:55:53 PM - HTTP Request: GET https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/harga_beras?select=%2A&tanggal=gte.2026-01-01&tanggal=lt.2027-01-01&offset=2000&limit=1000 "HTTP/2 200 OK"
03/03/2026 01:55:54 PM - HTTP Request: GET https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/harga_beras?select=%2A&tanggal=gte.2026-01-01&tanggal=lt.2027-01-01&offset=3000&limit=1000 "HTTP/2 200 OK"
03/03/2026 01:55:54 PM - HTTP Request: GET https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/harga_beras?select=%2A&tanggal=gte.2026-01-01&tanggal=lt.2027-01-01&offset=4000&limit=1000 "HTTP/2 200 OK"
03/03/20

Jumlah data untuk tahun 2026 yang diambil: 8621


,id,kode_kab_kota,tanggal,variant_id,harga,tipe_harga_id
0,122461,3501,2026-01-02,1,12300.0,2
1,122462,3503,2026-01-02,1,13000.0,2
2,122463,3504,2026-01-02,1,12650.0,2
3,122464,3505,2026-01-02,1,12750.0,2
4,122465,3506,2026-01-02,1,12500.0,2
...,...,...,...,...,...,...
8616,245646,3578,2026-02-26,2,15500.0,1
8617,245647,3579,2026-02-26,1,12800.0,1
8618,245648,3579,2026-02-26,2,14750.0,1
8619,245649,1,2026-02-26,1,12883.0,1


In [61]:
last_id_group = supabase.rpc("last_outlier_ids").execute().data[0]["last_group_id"]
last_id_detail = supabase.rpc("last_outlier_ids").execute().data[0]["last_detail_id"]

print(f"Last id group {last_id_group} \n last id detail {last_id_detail}")

03/03/2026 01:55:55 PM - HTTP Request: POST https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/rpc/last_outlier_ids "HTTP/2 200 OK"
03/03/2026 01:55:55 PM - HTTP Request: POST https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/rpc/last_outlier_ids "HTTP/2 200 OK"


Last id group 539 
 last id detail 13181


## detect outlier

In [62]:
def detect_iqr_outlier(df):
    df = df.copy()
    df["is_outlier"] = False

    group_cols = ["tahun", "variant_id", "tipe_harga_id", "kode_kab_kota"]

    for keys, g in df.groupby(group_cols):
        print("group is = ", group_cols)
        print("Group:", keys)
        print("Jumlah data:", len(g))

        q1 = g["harga"].quantile(0.25)
        q3 = g["harga"].quantile(0.75)
        iqr = q3 - q1

        print("Q1:", q1, "Q3:", q3, "IQR:", iqr)

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        print("Lower:", lower, "Upper:", upper)
        print("-"*40)

        idx = g[(g["harga"] < lower) | (g["harga"] > upper)].index
        df.loc[idx, "is_outlier"] = True

    return df



data_harga_beras["tanggal"] = pd.to_datetime(data_harga_beras["tanggal"])
data_harga_beras["tahun"] = data_harga_beras["tanggal"].dt.year
data_harga_beras = detect_iqr_outlier(data_harga_beras)

# ===============================
# INISIALISASI HOLIDAYS INDONESIA
# ===============================
id_holidays = holidays.Indonesia(years=range(data_harga_beras["tahun"].min(), data_harga_beras["tahun"].max() + 1))


group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2026), np.int64(1), np.int64(1), np.int64(1))
Jumlah data: 36
Q1: 12824.25 Q3: 12831.0 IQR: 6.75
Lower: 12814.125 Upper: 12841.125
----------------------------------------
group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2026), np.int64(1), np.int64(1), np.int64(3501))
Jumlah data: 36
Q1: 12824.25 Q3: 12831.0 IQR: 6.75
Lower: 12814.125 Upper: 12841.125
----------------------------------------
group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2026), np.int64(1), np.int64(1), np.int64(3502))
Jumlah data: 36
Q1: 12809.0 Q3: 12831.0 IQR: 22.0
Lower: 12776.0 Upper: 12864.0
----------------------------------------
group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2026), np.int64(1), np.int64(1), np.int64(3503))
Jumlah data: 36
Q1: 12809.0 Q3: 12831.0 IQR: 22.0
Lower: 12776.0 Upper: 12864.0
------

## Grouping Outlier 

In [63]:
def cluster_outlier(df, max_hari=14):
    """
    Membentuk cluster outlier berdasarkan tanggal berurutan
    dan maksimal durasi cluster = max_hari
    
    Cluster dibuat per kombinasi:
    tahun, variant_id, tipe_harga_id
    """

    df = df.copy()
    df["tanggal"] = pd.to_datetime(df["tanggal"])
    df["tahun"] = df["tanggal"].dt.year

    # ==============================
    # FILTER OUTLIER
    # ==============================
    out = df[df["is_outlier"] == True].copy()

    if out.empty:
        df["cluster_id"] = pd.NA
        return pd.DataFrame(), df

    out = out.sort_values(
        ["tahun", "variant_id", "tipe_harga_id", "tanggal"]
    )

    # ==============================
    # BENTUK CLUSTER DENGAN LOOP
    # ==============================
    cluster_list = []

    for keys, group in out.groupby(
        ["tahun", "variant_id", "tipe_harga_id"]
    ):
        group = group.sort_values("tanggal").copy()

        cluster_local = 1
        start_date = group.iloc[0]["tanggal"]
        prev_date = start_date

        clusters = [cluster_local]

        for i in range(1, len(group)):
            current_date = group.iloc[i]["tanggal"]

            selisih = (current_date - prev_date).days
            durasi = (current_date - start_date).days + 1

            # Putus cluster jika:
            # 1. Tidak berurutan (>1 hari)
            # 2. Durasi sudah melebihi max_hari
            if selisih > 1 or durasi > max_hari:
                cluster_local += 1
                start_date = current_date

            clusters.append(cluster_local)
            prev_date = current_date

        group["cluster_local"] = clusters
        cluster_list.append(group)

    out = pd.concat(cluster_list)

    # ==============================
    # SUMMARY CLUSTER
    # ==============================
    df_out = (
        out.groupby(
            ["tahun", "variant_id", "tipe_harga_id", "cluster_local"]
        )
        .agg(
            start_date=("tanggal", "min"),
            end_date=("tanggal", "max"),
            jumlah_data=("tanggal", "count")
        )
        .reset_index()
    )

    df_out["durasi_hari"] = (
        df_out["end_date"] - df_out["start_date"]
    ).dt.days + 1

    # ==============================
    # GLOBAL CLUSTER ID
    # ==============================
    df_out = df_out.sort_values(
        ["tahun", "variant_id", "tipe_harga_id", "start_date"]
    ).reset_index(drop=True)

    df_out["cluster_id"] = df_out.index + 1

    # ==============================
    # MAP KE OUTLIER
    # ==============================
    out = out.merge(
        df_out[
            ["tahun", "variant_id", "tipe_harga_id",
             "cluster_local", "cluster_id"]
        ],
        on=["tahun", "variant_id", "tipe_harga_id", "cluster_local"],
        how="left"
    )

    # ==============================
    # MERGE KE FULL DATA
    # ==============================
    df_full = df.merge(
        out[
            ["kode_kab_kota", "tanggal",
             "variant_id", "tipe_harga_id", "cluster_id"]
        ],
        on=["kode_kab_kota", "tanggal",
            "variant_id", "tipe_harga_id"],
        how="left"
    )

    df_full["cluster_id"] = df_full["cluster_id"].astype("Int64")

    return df_out, df_full

df_out, df_full = cluster_outlier(data_harga_beras)


In [64]:
df_out

,tahun,variant_id,tipe_harga_id,cluster_local,start_date,end_date,jumlah_data,durasi_hari,cluster_id
0,2026,1,1,1,2026-02-02,2026-02-06,110,5,1
1,2026,1,1,2,2026-02-09,2026-02-10,44,2,2
2,2026,1,1,3,2026-02-20,2026-02-20,36,1,3
3,2026,1,1,4,2026-02-23,2026-02-26,144,4,4
4,2026,1,2,1,2026-01-02,2026-01-15,136,14,5
5,2026,1,2,2,2026-01-16,2026-01-27,45,12,6
6,2026,1,2,3,2026-01-30,2026-01-30,2,1,7
7,2026,1,2,4,2026-02-02,2026-02-09,10,8,8
8,2026,1,2,5,2026-02-15,2026-02-17,5,3,9
9,2026,1,2,6,2026-02-19,2026-02-19,1,1,10


In [65]:
df_full.groupby('is_outlier').size()

is_outlier
False    7299
True     1322
dtype: int64

## Give information outlier

In [66]:
from gnews import GNews
import urllib.parse

def get_gnews_by_date(start_date, end_date,
                      language='id', country='ID'):
    
    keyword_raw = "harga OR impor beras OR logistik OR pangan OR cuaca ekstrim jawa timur"
    keyword = urllib.parse.quote(keyword_raw)
    # ✅ Pastikan format tuple
    if not isinstance(start_date, tuple):
        start_date = (start_date.year, start_date.month, start_date.day)

    if not isinstance(end_date, tuple):
        end_date = (end_date.year, end_date.month, end_date.day)
    
    google_news = GNews(
        language=language,
        country=country
    )
    
    google_news.start_date = start_date
    google_news.end_date = end_date
    
    news = google_news.get_news(keyword)
    
    # print("DEBUG jumlah raw news:", len(news))  # tambahkan debug
    
    data = []
    for article in news:
        data.append({
            "tanggal": article.get('published date'),
            "judul": article.get('title'),
            "sumber": article.get('publisher', {}).get('title'),
            "url": article.get('url')
        })
    
    df_news = pd.DataFrame(data)

    kalimat = df_to_sentences(df_news)
    
    # print("Jumlah berita: \n", kalimat)
    
    return kalimat

def df_to_sentences(df):
    sentences = []
    
    for _, row in df.iterrows():
        kalimat = (
            f"Pada {row['tanggal']}, media {row['sumber']} "
            f"memberitakan: \"{row['judul']}\"."
        )
        sentences.append(kalimat)
    
    return sentences

# ===============================
# FUNGSI UNTUK DETEKSI HARI LIBUR DI RENTANG
# ===============================
def get_holidays_in_range(start_date, end_date, buffer_days=7):
    """
    Deteksi hari libur dalam rentang tanggal, plus buffer sebelum/sesudah
    """
    holiday_list = []
    
    # Extend range dengan buffer
    extended_start = start_date - timedelta(days=buffer_days)
    extended_end = end_date + timedelta(days=buffer_days)
    
    current = extended_start
    while current <= extended_end:
        if current in id_holidays:
            holiday_list.append({
                'date': current,
                'name': id_holidays[current]
            })
        current += timedelta(days=1)
    
    return holiday_list



import re
from datetime import datetime

def format_news_item(news_text):
    """
    Parsing string berita menjadi format:
    - Menurut media X, pada Tanggal, Judul
    """

    # Contoh format:
    # Pada Sat, 10 Apr 2021 07:00:00 GMT, media Ngopibareng.id memberitakan: "Judul - Ngopibareng.id"

    pattern = r"Pada .*?, (\d{2} \w{3} \d{4}).*?media (.*?) memberitakan: \"(.*?)\""
    match = re.search(pattern, news_text)

    if match:
        tanggal = match.group(1)
        media = match.group(2)
        judul = match.group(3).split(" - ")[0]  # buang suffix nama media

        return f"- Menurut media {media}, pada {tanggal}, {judul}"
    
    return None


def ask_event_indonesia(start_date, end_date):

    holidays_found = get_holidays_in_range(start_date, end_date, buffer_days=7)
    berita = get_gnews_by_date(start_date, end_date)
    # print(berita)
    output_lines = []

    # ===============================
    # HOLIDAY SECTION
    # ===============================
    if holidays_found:
        output_lines.append("Berdekatan dengan hari besar:")

        for h in holidays_found:
            tanggal = h['date'].strftime('%d-%m-%Y')
            nama = h['name']
            output_lines.append(f"- {nama} ({tanggal})")


    # ===============================
    # NEWS SECTION
    # ===============================
    formatted_news = []


    for item in berita:
        formatted = format_news_item(item)
        if formatted:
            formatted_news.append(formatted)

    if formatted_news:
        if output_lines:
            output_lines.append("")  # spasi antar section
        output_lines.append("Terjadi peristiwa:")
        output_lines.extend(formatted_news)

    



    print(f"total berita {len(formatted_news)} | holiday {len(output_lines)}")
    print(f"hari besar \n {output_lines}")
    print(f"news \n {formatted_news}")

    # ===============================
    # FALLBACK
    # ===============================
    if not output_lines:
        return "Tidak ditemukan peristiwa dalam periode tersebut."

    return "\n".join(output_lines)


In [67]:
import time

def analyze_df_events(df_group, df_detail, sample_n=None, delay=2, error_delay=5):
    """
    Tambahkan kolom 'deskripsi' berdasarkan start_date & end_date.

    Parameters:
        df_group   : dataframe cluster (start_date, end_date, cluster_id)
        df_detail  : dataframe detail (is_outlier, id, cluster_id)
        sample_n   : jumlah baris diproses (None = semua)
        delay      : jeda normal antar request (detik)
        error_delay: jeda jika error (detik)
    """

    if sample_n == 999:
        sample_n = len(df_detail[df_detail["is_outlier"] == True])

    df = df_group.copy()

    df['start_date'] = pd.to_datetime(df['start_date'])
    df['end_date']   = pd.to_datetime(df['end_date'])

    target_df = df.head(sample_n) if sample_n else df

    total = len(target_df)

    print(f"\nTotal yang diproses: {total} baris\n")

    for i, (idx, row) in enumerate(target_df.iterrows(), 1):

        print(f" id {idx} [{i}/{total}] {row['start_date'].date()} → {row['end_date'].date()} ...", end=" ")

        try:
            analisis = ask_event_indonesia(row['start_date'], row['end_date'])
            df.loc[idx, 'deskripsi'] = analisis
            # print("✅")

            # delay normal
            if i < total:
                time.sleep(delay)

        except Exception as e:
            df.loc[idx, 'deskripsi'] = f"ERROR: {e}"
            print(f"❌ {e}")

            # delay lebih lama jika error
            time.sleep(error_delay)

    # ===============================
    # PREPARE OUTPUT
    # ===============================

    # ===============================
    # OUTLIER DETAIL
    # ===============================
    df_outlier_detail = (
        df_detail[df_detail["is_outlier"] == True][['id', 'cluster_id']]
        .reset_index(drop=True)
    )

    df_outlier_detail.insert(0, 'row_id', range(1, len(df_outlier_detail) + 1))


    # ===============================
    # OUTLIER GROUP
    # ===============================
    df_outlier_group = (
        df[['cluster_id', 'deskripsi']]
        .reset_index(drop=True)
    )

    df_outlier_group.insert(0, 'row_id', range(1, len(df_outlier_group) + 1))

    df_outlier_detail = df_outlier_detail.reset_index(drop=True)
    df_outlier_group  = df_outlier_group.reset_index(drop=True)

    return df_outlier_detail, df_outlier_group


# # ===============================
# # CONTOH PENGGUNAAN
# # ===============================

# # Test 3 baris dulu sebelum proses semua
# outlier_detail, outlier_group = analyze_df_events(df_out, df_full, sample_n=1)
# # print(outlier_group.iloc[0]['deskripsi'])

In [68]:
def analyze_in_batches(df_group, df_detail, batch_size=50, delay_between_batch=10):
    
    total = len(df_group)
    results_detail = []
    results_group = []
    
    print(f"Total cluster: {total}")
    print(f"Batch size: {batch_size}")
    print(f"Total batch: {(total // batch_size) + 1}\n")
    
    for start in range(0, total, batch_size):
        
        end = start + batch_size
        print(f"\n===== PROSES BATCH {start} - {min(end, total)} =====")
        
        df_batch = df_group.iloc[start:end]
        
        detail, group = analyze_df_events(
            df_batch,
            df_detail,
            sample_n=None,
            delay=2,
            error_delay=5
        )
        
        results_detail.append(detail)
        results_group.append(group)
        
        # Delay antar batch supaya API aman
        if end < total:
            print(f"\n⏳ Tunggu {delay_between_batch} detik sebelum batch berikutnya...\n")
            time.sleep(delay_between_batch)
    
    final_detail = pd.concat(results_detail, ignore_index=True)
    final_group  = pd.concat(results_group, ignore_index=True)
    
    return final_detail, final_group

outlier_detail, outlier_group = analyze_in_batches(
    df_out,
    df_full,
    batch_size=50,
    delay_between_batch=15
)

Total cluster: 44
Batch size: 50
Total batch: 1


===== PROSES BATCH 0 - 44 =====

Total yang diproses: 44 baris

 id 0 [1/44] 2026-02-02 → 2026-02-06 ... total berita 6 | holiday 7
hari besar 
 ['Terjadi peristiwa:', '- Menurut media CNN Indonesia, pada 06 Feb 2026, Peringatan BMKG: Cuaca Ekstrem Mengintai Indonesia Sepekan ke Depan', '- Menurut media CNN Indonesia, pada 06 Feb 2026, Dampak Krisis Iklim, Hujan di Indonesia Makin Ekstrem', '- Menurut media ANTARA News Megapolitan, pada 02 Feb 2026, 19 rumah warga rusak akibat hujan dan angin kencang di Makassar', '- Menurut media RRI.co.id, pada 02 Feb 2026, BMKG: Sebagian Wilayah Aceh Malam Ini Diprediksi Hujan', '- Menurut media Qoo10.co.id, pada 05 Feb 2026, Hujan Dominasi Jawa Tengah 4-5 Feb 2026, Waspada Petir di Wilayah Banyumas & Cilacap', '- Menurut media Qoo10.co.id, pada 05 Feb 2026, Waspada Hujan Lebat & Angin Kencang di Jawa Barat 5 Feb 2026, Cek Prakiraan Terbaru BMKG']
news 
 ['- Menurut media CNN Indonesia, pada 06 Feb 2

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 2
hari besar 
 ['Berdekatan dengan hari besar:', '- Lunar New Year (estimated) (17-02-2026)']
news 
 []
 id 3 [4/44] 2026-02-23 → 2026-02-26 ... total berita 8 | holiday 12
hari besar 
 ['Berdekatan dengan hari besar:', '- Lunar New Year (estimated) (17-02-2026)', '', 'Terjadi peristiwa:', '- Menurut media Nusadaily, pada 23 Feb 2026, Awas! BMKG Peringatkan Malang Raya dan Jatim Waspada Cuaca Ekstrem', '- Menurut media Beritajateng.tv, pada 24 Feb 2026, Kamis Berpotensi Diguyur Hujan Lebat, BMKG Imbau Masyarakat Tingkatkan Kewaspadaan', '- Menurut media Kompas.tv, pada 24 Feb 2026, Peringatan Dini BMKG 25 26 Februari 2026: Hujan Lebat Mengintai Sejumlah Provinsi, Ini Daftarnya', '- Menurut media Jatimtimes, pada 24 Feb 2026, Bupati Malang Tinjau Dampak Bencana Puting Beliung: Rusak 15 Rumah, Kerugian Puluhan Juta', '- Menurut media Beritajateng.tv, pada 24 Feb 2026, Waspada Hujan Lebat, BMKG Rilis Prakiraan Cuaca Rabu 25 Februari 2026', '- Menurut media mandali

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 0
hari besar 
 []
news 
 []
 id 7 [8/44] 2026-02-02 → 2026-02-09 ... total berita 7 | holiday 8
hari besar 
 ['Terjadi peristiwa:', '- Menurut media Kompas.tv, pada 08 Feb 2026, BMKG Sebut Ada Peringatan Dini Cuaca Ekstrem 9-10 Februari 2026, Waspada Banjir!', '- Menurut media ANTARA News Megapolitan, pada 02 Feb 2026, 19 rumah warga rusak akibat hujan dan angin kencang di Makassar', '- Menurut media RRI.co.id, pada 09 Feb 2026, BMKG: Waspada, Aceh Malam Ini Kembali Diguyur Hujan', '- Menurut media Kompas.tv, pada 08 Feb 2026, Prakiraan Cuaca BMKG Hari Ini Minggu 8 Februari 2026: Jawa dan NTT Waspada Hujan Lebat', '- Menurut media RRI.co.id, pada 02 Feb 2026, BMKG: Sebagian Wilayah Aceh Malam Ini Diprediksi Hujan', '- Menurut media Qoo10.co.id, pada 05 Feb 2026, Hujan Dominasi Jawa Tengah 4-5 Feb 2026, Waspada Petir di Wilayah Banyumas & Cilacap', '- Menurut media Qoo10.co.id, pada 05 Feb 2026, Waspada Hujan Lebat & Angin Kencang di Jawa Barat 5 Feb 2026, Cek P

...

total berita 0 | holiday 0
hari besar 
 []
news 
 []
 id 41 [42/44] 2026-02-06 → 2026-02-06 ... 

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 3 | holiday 4
hari besar 
 ['Terjadi peristiwa:', '- Menurut media CNN Indonesia, pada 06 Feb 2026, Peringatan BMKG: Cuaca Ekstrem Mengintai Indonesia Sepekan ke Depan', '- Menurut media CNN Indonesia, pada 06 Feb 2026, Dampak Krisis Iklim, Hujan di Indonesia Makin Ekstrem', '- Menurut media Siaga Indonesia, pada 06 Feb 2026, Wali Kota Eri Imbau Warga Siaga Hadapi Cuaca Ekstrim, Komisi A: CC Room dan Call Centre 112 Harus Standby 24 Jam']
news 
 ['- Menurut media CNN Indonesia, pada 06 Feb 2026, Peringatan BMKG: Cuaca Ekstrem Mengintai Indonesia Sepekan ke Depan', '- Menurut media CNN Indonesia, pada 06 Feb 2026, Dampak Krisis Iklim, Hujan di Indonesia Makin Ekstrem', '- Menurut media Siaga Indonesia, pada 06 Feb 2026, Wali Kota Eri Imbau Warga Siaga Hadapi Cuaca Ekstrim, Komisi A: CC Room dan Call Centre 112 Harus Standby 24 Jam']
 id 42 [43/44] 2026-02-14 → 2026-02-14 ... 

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 2
hari besar 
 ['Berdekatan dengan hari besar:', '- Lunar New Year (estimated) (17-02-2026)']
news 
 []
 id 43 [44/44] 2026-02-16 → 2026-02-26 ... total berita 11 | holiday 15
hari besar 
 ['Berdekatan dengan hari besar:', '- Lunar New Year (estimated) (17-02-2026)', '', 'Terjadi peristiwa:', '- Menurut media Nusadaily, pada 23 Feb 2026, Awas! BMKG Peringatkan Malang Raya dan Jatim Waspada Cuaca Ekstrem', '- Menurut media Sinergia Mediatama, pada 19 Feb 2026, Waspada Cuaca Ekstrem di Madiun, Sejumlah Rumah Rusak Diterjang Hujan dan Angin Kencang', '- Menurut media RRI.co.id, pada 16 Feb 2026, Cuaca Ekstrim Harga Cabai Melonjak Rp 45.000/Kg', '- Menurut media Beritajateng.tv, pada 24 Feb 2026, Kamis Berpotensi Diguyur Hujan Lebat, BMKG Imbau Masyarakat Tingkatkan Kewaspadaan', '- Menurut media VIVA Banyuwangi, pada 23 Feb 2026, Cuaca Ekstrem, KAI Daop 9 Jember Perkuat Infrastruktur dan Siagakan Petugas Ekstra', '- Menurut media Beritajateng.tv, pada 22 Feb 2026,

## Setup Coloumn

In [69]:
outlier_detail['id_harga_beras'] = outlier_detail['id']
outlier_detail['id'] = outlier_detail['row_id']+last_id_detail
outlier_detail['id_group'] = outlier_detail['cluster_id']+last_id_group
outlier_detail = outlier_detail[['id', 'id_harga_beras', 'id_group']]

# Menghapus duplikat berdasarkan kolom 'id_harga_beras' dan 'id_group', hanya menyisakan satu baris
outlier_detail = outlier_detail.drop_duplicates(subset=['id_harga_beras', 'id_group'], keep='first')

# Menampilkan hasil
print(f"Jumlah baris setelah duplikat dihapus: {outlier_detail.shape[0]}")
outlier_detail[outlier_detail['id_group'] == last_id_group+1]

Jumlah baris setelah duplikat dihapus: 1322


,id,id_harga_beras,id_group
418,13600,206097,540
419,13601,206098,540
420,13602,206099,540
421,13603,206100,540
422,13604,206101,540
...,...,...,...
868,14050,243565,540
869,14051,243566,540
870,14052,243567,540
871,14053,243568,540


In [70]:
outlier_group['id'] = outlier_group['cluster_id']+last_id_group
outlier_group = outlier_group[['id', 'deskripsi']]
outlier_group

,id,deskripsi
0,540,Terjadi peristiwa:\n- Menurut media CNN Indone...
1,541,Berdekatan dengan hari besar:\n- Lunar New Yea...
2,542,Berdekatan dengan hari besar:\n- Lunar New Yea...
3,543,Berdekatan dengan hari besar:\n- Lunar New Yea...
4,544,Berdekatan dengan hari besar:\n- New Year's Da...
5,545,Berdekatan dengan hari besar:\n- Isra' and Mi'...
6,546,Tidak ditemukan peristiwa dalam periode tersebut.
7,547,"Terjadi peristiwa:\n- Menurut media Kompas.tv,..."
8,548,Berdekatan dengan hari besar:\n- Lunar New Yea...
9,549,Berdekatan dengan hari besar:\n- Lunar New Yea...


## Insert Into Supabase

In [71]:
import math

def insert_large_data(table_name, data, batch_size=900):
    total = len(data)
    batches = math.ceil(total / batch_size)

    for i in range(batches):
        start = i * batch_size
        end = start + batch_size
        batch = data[start:end]

        response = supabase.table(table_name).insert(batch).execute()

        if response.data is None:
            print("Error:", response)
            break

        print(f"Batch {i+1}/{batches} berhasil ({len(batch)} rows)")

    print("Selesai insert.")

# ======================
# CONTOH PAKAI
# ======================
outlier_group_list = outlier_group.to_dict(orient="records")

insert_large_data("outlier_group", outlier_group_list)

03/03/2026 01:59:00 PM - HTTP Request: POST https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/outlier_group?columns=%22deskripsi%22%2C%22id%22 "HTTP/2 201 Created"


Batch 1/1 berhasil (44 rows)
Selesai insert.


In [72]:
outlier_detail_list = outlier_detail.to_dict(orient="records")

insert_large_data("outlier_detail", outlier_detail_list) 

03/03/2026 01:59:01 PM - HTTP Request: POST https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/outlier_detail?columns=%22id_harga_beras%22%2C%22id_group%22%2C%22id%22 "HTTP/2 201 Created"
03/03/2026 01:59:01 PM - HTTP Request: POST https://extlxiwpcbzqaalpopqn.supabase.co/rest/v1/outlier_detail?columns=%22id_harga_beras%22%2C%22id_group%22%2C%22id%22 "HTTP/2 201 Created"


Batch 1/2 berhasil (900 rows)
Batch 2/2 berhasil (422 rows)
Selesai insert.
